## 1. Configuración

In [ ]:
import os
REPS = list(range(2, 11))          # todas; o por bloques: [2, 3, 4] / [5, 6, 7] / [8, 9, 10]

CARPETA   = os.environ.get('CARPETA_ARTICULO', '/content/drive/MyDrive/Articulo_Grupal_EdgeIIoT')
NB_FUENTE = os.environ.get('NB_FUENTE', '/content/drive/MyDrive/Colab Notebooks/S10_Desarrollo_Semilla1.ipynb')
DIR_TRABAJO = os.environ.get('DIR_TRABAJO', '/content')   # aquí se escribe medir_modelo.py en cada corrida

COPIAR_CSV_LOCAL = True            # copia el CSV (1,2 GB) al disco de la VM una vez: lectura más rápida en cada REP
TIEMPO_MAX_CELDA_S = 3 * 3600      # si una celda pasa de 3 h se considera colgada
DETENER_SI_FALLA = True
FORZAR_ENTORNO_DISTINTO = False    # True solo si aceptas versiones distintas a las de REP-01 (quedará registrado)

## 2. Drive, dependencias y control del entorno (debe coincidir con REP-01)

In [ ]:
import sys, subprocess, json, re, time, shutil, platform, datetime
from importlib.metadata import version, PackageNotFoundError
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'papermill', 'imbalanced-learn', 'xgboost', 'kagglehub'], check=True)
import psutil, pandas as pd, nbformat, papermill as pm

def v(p):
    try: return version(p)
    except PackageNotFoundError: return 'no instalado'

def cpu_modelo():
    try:
        for l in open('/proc/cpuinfo'):
            if l.startswith('model name'):
                return l.split(':', 1)[1].strip()
    except Exception:
        pass
    return platform.processor()

def gpu_nombre():
    try:
        return subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                              capture_output=True, text=True).stdout.strip() or 'ninguna'
    except Exception:
        return 'ninguna'

actual = {'procesador': cpu_modelo(), 'nucleos_logicos': psutil.cpu_count(),
          'ram_total_gb': round(psutil.virtual_memory().total / 1024**3, 2),
          'acelerador_entrenamiento_tcn': gpu_nombre(), 'python': platform.python_version(),
          'numpy': v('numpy'), 'pandas': v('pandas'), 'scikit_learn': v('scikit-learn'),
          'imbalanced_learn': v('imbalanced-learn'), 'xgboost': v('xgboost'),
          'tensorflow': v('tensorflow'), 'keras': v('keras'), 'scipy': v('scipy')}

ficha1 = os.path.join(CARPETA, 'corridas', 'REP-01', 'ficha_REP-01.json')
ref = json.load(open(ficha1, encoding='utf-8'))['tabla_14_2']
comp = pd.DataFrame({'REP-01': {k: ref.get(k) for k in actual}, 'Esta sesión': actual})
comp['¿Igual?'] = [str(a) == str(b) for a, b in zip(comp['REP-01'], comp['Esta sesión'])]
display(comp)

criticos = ['acelerador_entrenamiento_tcn', 'python', 'numpy', 'pandas', 'scikit_learn',
            'imbalanced_learn', 'xgboost', 'tensorflow', 'keras', 'scipy']
distintos = [k for k in criticos if not comp.loc[k, '¿Igual?']]
if not str(actual['acelerador_entrenamiento_tcn']).upper().count('T4'):
    raise SystemExit('⛔ No hay GPU T4. Cambia el tipo de entorno a T4 con RAM alta y vuelve a ejecutar.')
if actual['ram_total_gb'] < 45:
    raise SystemExit('⛔ No es el entorno de RAM alta (REP-01 tenía 50,99 GB).')
if distintos and not FORZAR_ENTORNO_DISTINTO:
    raise SystemExit(f'⛔ Versiones distintas a REP-01: {distintos}. Revisa antes de seguir '
                     '(o pon FORZAR_ENTORNO_DISTINTO = True y se registrará en cada ficha).')
if not comp.loc['procesador', '¿Igual?']:
    print('⚠️ El procesador de la VM difiere del de REP-01. Se puede seguir: la comparación O₁ vs O₂ es '
          'dentro de cada repetición, pero queda registrado en la ficha y el consolidado lo señalará.')
print('✔ Entorno verificado')

Mounted at /content/drive


,REP-01,Esta sesión,¿Igual?
procesador,Intel(R) Xeon(R) CPU @ 2.00GHz,Intel(R) Xeon(R) CPU @ 2.00GHz,True
nucleos_logicos,8,8,True
ram_total_gb,50.99,50.99,True
acelerador_entrenamiento_tcn,Tesla T4,Tesla T4,True
python,3.13.16,3.13.16,True
numpy,2.4.6,2.4.6,True
pandas,2.2.3,2.2.3,True
scikit_learn,1.6.1,1.6.1,True
imbalanced_learn,0.14.2,0.14.2,True
xgboost,3.4.1,3.4.1,True


✔ Entorno verificado


## 3. Copia local del CSV

---



In [ ]:
os.environ['CARPETA_ARTICULO'] = CARPETA
csv_drive = os.path.join(CARPETA, 'DNN-EdgeIIoT-dataset.csv')
if COPIAR_CSV_LOCAL:
    csv_local = os.path.join(DIR_TRABAJO, 'DNN-EdgeIIoT-dataset.csv')
    if not (os.path.exists(csv_local) and os.path.getsize(csv_local) == os.path.getsize(csv_drive)):
        t0 = time.time(); shutil.copy(csv_drive, csv_local)
        print(f'CSV copiado en {time.time()-t0:.0f} s')
    os.environ['EDGE_CSV'] = csv_local       # la celda 1 del notebook S10 ya lee esta variable
else:
    os.environ['EDGE_CSV'] = csv_drive
print('EDGE_CSV =', os.environ['EDGE_CSV'])

CSV copiado en 20 s
EDGE_CSV = /content/DNN-EdgeIIoT-dataset.csv


## 4. Copia por repetición
Solo cambia dos líneas de infraestructura; la metodología queda intacta:
1. `REP = 1` → `REP = r` (celda 1).
2. `drive.mount(...)` se salta si Drive ya está montado (el kernel nuevo no puede pedir autorización).

In [ ]:
PATRON_REP = re.compile(r'^REP\s*=\s*\d+', re.M)
MONTAJE = "drive.mount('/content/drive')"

def preparar(rep):
    nb = nbformat.read(NB_FUENTE, as_version=4)
    con_rep = [c for c in nb.cells if c.cell_type == 'code' and PATRON_REP.search(c.source)]
    assert len(con_rep) == 1, 'No se encontró una única línea "REP = ..." en el notebook fuente'
    c = con_rep[0]
    c.source = PATRON_REP.sub(f'REP = {rep}', c.source, count=1)
    assert re.search(r'^MODO_PRUEBA\s*=\s*False', c.source, re.M), 'MODO_PRUEBA debe ser False'
    assert re.search(r'^REANUDAR\s*=\s*True', c.source, re.M), 'REANUDAR debe ser True'
    for c in nb.cells:
        if c.cell_type == 'code' and MONTAJE in c.source:
            c.source = c.source.replace(MONTAJE, MONTAJE + " if not os.path.isdir('/content/drive/MyDrive') else None")
    nb.metadata['kernelspec'] = {'name': 'python3', 'display_name': 'Python 3', 'language': 'python'}
    ruta = os.path.join(DIR_TRABAJO, f'S10_Desarrollo_REP-{rep:02d}.ipynb')
    nbformat.write(nb, ruta)
    return ruta

print('Celda 1 de la copia para REP-02:')
print([c for c in nbformat.read(preparar(2), as_version=4).cells if PATRON_REP.search(c.source)][0].source.splitlines()[2])

Celda 1 de la copia para REP-02:
REP = 2                                        # repetición r (1..10)


## 5. Ejecución (un kernel nuevo por repetición)

In [ ]:
BITACORA = os.path.join(CARPETA, 'corridas', 'bitacora_S11.csv')

def resumen(cod, minutos):
    f = json.load(open(os.path.join(CARPETA, 'corridas', cod, f'ficha_{cod}.json'), encoding='utf-8'))
    m = {r['Indicador (unidad)']: r for r in f['tabla_14_3']}
    f1 = m['Puntaje F1 macro (%)']; p = f['tabla_14_5'][0]
    return {'corrida': cod, 'semilla': f['tabla_14_2']['semilla'], 'fin': datetime.datetime.now().strftime('%d/%m %H:%M'),
            'minutos': round(minutos, 1), 'procesador': f['tabla_14_2']['procesador'],
            'F1 TCN': round(f1['O1 TCN'], 2), 'F1 RF': round(f1['O2 RF'], 2), 'F1 XGB': round(f1['O2 XGBoost'], 2),
            'k': p['k'], 'T': p['T'], 'RF': f"{p['RF: árboles']}/{p['RF: profundidad']}",
            'XGB': f"{p['XGB: árboles']}/{p['XGB: profundidad']}"}

for rep in REPS:
    cod = f'REP-{rep:02d}'
    dir_run = os.path.join(CARPETA, 'corridas', cod)
    if os.path.exists(os.path.join(dir_run, f'ficha_{cod}.json')):
        print(f'{cod}: ya tiene ficha → se omite'); continue
    os.makedirs(dir_run, exist_ok=True)
    print(f'\n▶ {cod} (semilla {1000 + rep}) — inicio {datetime.datetime.now():%H:%M}')
    t0 = time.time()
    try:
        pm.execute_notebook(preparar(rep), os.path.join(dir_run, f'S10_Desarrollo_{cod}_ejecutado.ipynb'),
                            kernel_name='python3', cwd=DIR_TRABAJO, progress_bar=True, log_output=False,
                            execution_timeout=TIEMPO_MAX_CELDA_S)
    except Exception as e:
        print(f'✖ {cod} falló: {type(e).__name__}: {str(e)[:1500]}')
        print(f'   Revisa {cod}/S10_Desarrollo_{cod}_ejecutado.ipynb para ver la celda que falló.')
        if DETENER_SI_FALLA:
            raise
        continue
    fila = resumen(cod, (time.time() - t0) / 60)
    pd.DataFrame([fila]).to_csv(BITACORA, mode='a', header=not os.path.exists(BITACORA), index=False)
    print('✔', fila)


▶ REP-02 (semilla 1002) — inicio 14:50


Executing:   0%|          | 0/31 [00:00<?, ?cell/s]

✔ {'corrida': 'REP-02', 'semilla': 1002, 'fin': '11/11 15:27', 'minutos': 37.1, 'procesador': 'Intel(R) Xeon(R) CPU @ 2.00GHz', 'F1 TCN': 68.94, 'F1 RF': 90.5, 'F1 XGB': 89.9, 'k': 25, 'T': 2000, 'RF': '25/15', 'XGB': '50/4'}

▶ REP-03 (semilla 1003) — inicio 15:27


Executing:   0%|          | 0/31 [00:00<?, ?cell/s]

✔ {'corrida': 'REP-03', 'semilla': 1003, 'fin': '11/11 16:03', 'minutos': 35.8, 'procesador': 'Intel(R) Xeon(R) CPU @ 2.00GHz', 'F1 TCN': 69.86, 'F1 RF': 90.32, 'F1 XGB': 90.47, 'k': 25, 'T': 2000, 'RF': '25/15', 'XGB': '50/4'}

▶ REP-04 (semilla 1004) — inicio 16:03


Executing:   0%|          | 0/31 [00:00<?, ?cell/s]

... (9 outputs left out)

Executing:   0%|          | 0/31 [00:00<?, ?cell/s]

✔ {'corrida': 'REP-09', 'semilla': 1009, 'fin': '11/11 19:42', 'minutos': 35.0, 'procesador': 'Intel(R) Xeon(R) CPU @ 2.00GHz', 'F1 TCN': 68.5, 'F1 RF': 90.25, 'F1 XGB': 89.63, 'k': 25, 'T': 2000, 'RF': '25/15', 'XGB': '50/4'}

▶ REP-10 (semilla 1010) — inicio 19:42


Executing:   0%|          | 0/31 [00:00<?, ?cell/s]

✔ {'corrida': 'REP-10', 'semilla': 1010, 'fin': '11/11 20:19', 'minutos': 37.0, 'procesador': 'Intel(R) Xeon(R) CPU @ 2.00GHz', 'F1 TCN': 68.26, 'F1 RF': 90.38, 'F1 XGB': 89.88, 'k': 25, 'T': 2000, 'RF': '25/15', 'XGB': '50/4'}


## 6. Estado de las 10 repeticiones

In [ ]:
filas = []
for r in range(1, 11):
    cod = f'REP-{r:02d}'
    if os.path.exists(os.path.join(CARPETA, 'corridas', cod, f'ficha_{cod}.json')):
        filas.append(resumen(cod, float('nan')))
    else:
        filas.append({'corrida': cod, 'semilla': 1000 + r, 'fin': 'PENDIENTE'})
estado = pd.DataFrame(filas).drop(columns=['minutos', 'fin'], errors='ignore')
display(estado)
faltan = estado['corrida'][estado.get('F1 TCN').isna()].tolist() if 'F1 TCN' in estado else estado['corrida'].tolist()
print('Faltan:', faltan or 'ninguna — listo para S11_Consolidacion_Estadistica.ipynb')

,corrida,semilla,procesador,F1 TCN,F1 RF,F1 XGB,k,T,RF,XGB
0,REP-01,1001,Intel(R) Xeon(R) CPU @ 2.00GHz,68.46,90.09,90.06,25,2000,25/15,50/4
1,REP-02,1002,Intel(R) Xeon(R) CPU @ 2.00GHz,68.94,90.50,89.90,25,2000,25/15,50/4
2,REP-03,1003,Intel(R) Xeon(R) CPU @ 2.00GHz,69.86,90.32,90.47,25,2000,25/15,50/4
3,REP-04,1004,Intel(R) Xeon(R) CPU @ 2.00GHz,68.31,90.47,90.26,25,2000,25/15,50/4
4,REP-05,1005,Intel(R) Xeon(R) CPU @ 2.00GHz,68.81,90.30,89.99,25,2000,25/15,50/4
5,REP-06,1006,Intel(R) Xeon(R) CPU @ 2.00GHz,68.88,90.32,90.06,25,2000,25/15,50/4
6,REP-07,1007,Intel(R) Xeon(R) CPU @ 2.00GHz,68.38,90.31,89.94,25,2000,25/15,50/4
7,REP-08,1008,Intel(R) Xeon(R) CPU @ 2.00GHz,66.93,90.24,90.03,25,2000,25/15,50/4
8,REP-09,1009,Intel(R) Xeon(R) CPU @ 2.00GHz,68.50,90.25,89.63,25,2000,25/15,50/4
9,REP-10,1010,Intel(R) Xeon(R) CPU @ 2.00GHz,68.26,90.38,89.88,25,2000,25/15,50/4


Faltan: ninguna — listo para S11_Consolidacion_Estadistica.ipynb
